In [1]:

# Install required libraries
!pip install pandas numpy scikit-learn -q

# Import libraries
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

# Movie details
movies = pd.DataFrame({
    'movieId': [1, 2, 3, 4, 5, 6, 7, 8],
    'title': [
        'The Avengers',
        'Iron Man',
        'Titanic',
        'The Notebook',
        'Interstellar',
        'Inception',
        'Toy Story',
        'Finding Nemo'
    ],
    'genres': [
        'Action Adventure Sci-Fi',
        'Action Adventure Sci-Fi',
        'Romance Drama',
        'Romance Drama',
        'Sci-Fi Drama',
        'Action Sci-Fi Thriller',
        'Animation Comedy Family',
        'Animation Adventure Family'
    ]
})

# User ratings
ratings = pd.DataFrame({
    'userId': [
        1, 1, 1, 1, 2, 2, 2, 2,
        3, 3, 3, 3, 4, 4, 4, 4
    ],
    'movieId': [
        1, 2, 5, 6, 3, 4, 5, 8,
        1, 2, 6, 7, 3, 4, 7, 8
    ],
    'rating': [
        5, 4, 5, 4, 5, 4, 3, 5,
        4, 5, 4, 3, 4, 5, 4, 5
    ]
})

print("Movie Dataset:")
display(movies)

print("User Ratings Dataset:")
display(ratings)


Movie Dataset:


,movieId,title,genres
0,1,The Avengers,Action Adventure Sci-Fi
1,2,Iron Man,Action Adventure Sci-Fi
2,3,Titanic,Romance Drama
3,4,The Notebook,Romance Drama
4,5,Interstellar,Sci-Fi Drama
5,6,Inception,Action Sci-Fi Thriller
6,7,Toy Story,Animation Comedy Family
7,8,Finding Nemo,Animation Adventure Family


User Ratings Dataset:


,userId,movieId,rating
0,1,1,5
1,1,2,4
2,1,5,5
3,1,6,4
4,2,3,5
5,2,4,4
6,2,5,3
7,2,8,5
8,3,1,4
9,3,2,5


In [3]:

# Convert movie genres into numerical features
tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(movies['genres'])

# Calculate similarity between movies
similarity_matrix = cosine_similarity(genre_matrix)

# Recommendation function
def recommend_by_genre(movie_title, top_n=3):
    if movie_title not in movies['title'].values:
        print("Movie not found. Please enter a valid title.")
        return

    movie_index = movies.index[
        movies['title'] == movie_title
    ][0]

    scores = list(enumerate(similarity_matrix[movie_index]))

    # Sort by similarity, excluding the selected movie
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [
        item for item in scores
        if item[0] != movie_index
    ][:top_n]

    result = movies.iloc[[i for i, score in scores]][
        ['title', 'genres']
    ].copy()

    result['similarity'] = [
        round(float(score), 3) for i, score in scores
    ]

    print(f"Movies similar to '{movie_title}':")
    display(result.reset_index(drop=True))

# Test the system
recommend_by_genre('The Avengers')


Movies similar to 'The Avengers':


,title,genres,similarity
0,Iron Man,Action Adventure Sci-Fi,1.000
1,Inception,Action Sci-Fi Thriller,0.640
2,Interstellar,Sci-Fi Drama,0.513


In [4]:

# Create a user-movie rating matrix
rating_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)

# Calculate similarity between users
user_similarity = cosine_similarity(rating_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=rating_matrix.index,
    columns=rating_matrix.index
)

# Recommendation function
def recommend_by_ratings(user_id, top_n=3):
    if user_id not in rating_matrix.index:
        print("User not found.")
        return

    # Get similarity scores for this user
    similar_users = user_similarity_df.loc[user_id].drop(
        user_id
    ).sort_values(ascending=False)

    # Calculate weighted ratings from other users
    scores = {}
    weights = {}

    for other_user, similarity in similar_users.items():
        if similarity <= 0:
            continue

        other_ratings = rating_matrix.loc[other_user]
        unseen = rating_matrix.loc[user_id] == 0

        for movie_id in rating_matrix.columns[unseen]:
            rating = other_ratings[movie_id]

            if rating > 0:
                scores[movie_id] = (
                    scores.get(movie_id, 0)
                    + similarity * rating
                )
                weights[movie_id] = (
                    weights.get(movie_id, 0)
                    + similarity
                )

    # Calculate weighted average scores
    recommendations = []
    for movie_id in scores:
        predicted_rating = scores[movie_id] / weights[movie_id]
        recommendations.append((movie_id, predicted_rating))

    recommendations.sort(key=lambda x: x[1], reverse=True)
    recommendations = recommendations[:top_n]

    if not recommendations:
        print("No recommendations available for this user.")
        return

    result = pd.DataFrame([
        {
            'Movie': movies.loc[
                movies['movieId'] == movie_id, 'title'
            ].iloc[0],
            'Predicted Rating': round(float(score), 2)
        }
        for movie_id, score in recommendations
    ])

    print(f"Recommendations for User {user_id}:")
    display(result)

# Test the system
recommend_by_ratings(1)


Recommendations for User 1:


,Movie,Predicted Rating
0,Titanic,5.0
1,Finding Nemo,5.0
2,The Notebook,4.0


In [ ]:

# Content-based recommendation
movie_name = input("Enter a movie title: ")
recommend_by_genre(movie_name, top_n=3)

# Collaborative filtering
user = int(input("Enter user ID (1 to 4): "))
recommend_by_ratings(user, top_n=3)


Enter a movie title: Inception
Movies similar to 'Inception':


,title,genres,similarity
0,The Avengers,Action Adventure Sci-Fi,0.640
1,Iron Man,Action Adventure Sci-Fi,0.640
2,Interstellar,Sci-Fi Drama,0.458
